# Running inference on data

This notebook shows how to load a model that a trainer has already produced and use it for prediction, both from Python and from a config file.

`ClassifierInference` and `RegressionInference` load an exported model and expose a single `predict` method. They are prediction-only: there is no `fit`, and they do not plug into training tools such as `GridSearchCV`. Loading trusts the artifact, exactly as the trainers' own snapshot loading does, so only load models you produced or otherwise trust.

`predict` accepts a NumPy array or a torch tensor and returns predicted labels (classifier) or values (regressor). Class probabilities are not currently exposed by these classes.

**Before you run this notebook, delete the results from previous runs of any example notebooks to avoid getting 'file exists' errors.**

## 1. Train a model to run inference on

Inference needs an exported model. We borrow the binary random-forest run from `simpletrainer_examples.ipynb`: load its config, build the dataset, fit, and snapshot. `save_snapshot` writes `config.yaml` and `model.skops` under the trainer's `output_path`; inference only needs the `model.skops` file.

Like the other example notebooks, this runs top to bottom against a clean `training/` directory.

In [ ]:
import torch
import yaml
from torch.utils.data import random_split

from GalaxySpectrumClassifier import SimpleTrainer, TabularDataset

with open("../configs/binary_classsifier_simple_example.yaml") as f:
    config = yaml.safe_load(f)

dataset = TabularDataset.from_config(config["dataset"])
train_dataset, test_dataset = random_split(
    dataset, [0.8, 0.2], generator=torch.Generator().manual_seed(42)
)

trainer = SimpleTrainer.from_config(config["trainer"])
trainer.fit(train_dataset)
trainer.save_snapshot("trained_random_forest")

## 2. Load the model with `ClassifierInference`

`ClassifierInference` needs three things: the path to the exported model, its `model_format`, and the `task`. A `SimpleTrainer` export is a single self-contained scikit-learn artifact, so the format is `skops` and the path is the `model.skops` file.

In [ ]:
from GalaxySpectrumClassifier import ClassifierInference

model_file = (
    "../training/binaryclassifier_simple_example/trained_random_forest/model.skops"
)
classifier = ClassifierInference(
    model_file,
    model_format="skops",
    task="binary-classification",
)

### Predict

`predict` takes a feature array. `to_xy` turns a dataset (or a split) into the `X, y` arrays the underlying estimator expects; here we score the held-out test split.

In [ ]:
from GalaxySpectrumClassifier import to_xy

X_test, y_test = to_xy(test_dataset)

predictions = classifier.predict(X_test)
accuracy = (predictions == y_test).mean()
accuracy

A single sample works the same way — pass a 2D array with one row.

In [ ]:
prediction = classifier.predict(X_test[:1])
prediction, y_test[:1]

## Predict probabilities that a sample belongs to a class
This can be done with the `predict_proba` method that is known from scikit-learn classifiers. It returns an array of shape `(n_samples, n_classes)` in which the columns correspond the the class indices, e.g.:

```python
   classifier.classes_
```

`array([0, 1])`

```python
   classifier.predict_proba(X_new)            
```

`array([[0.20, 0.80], [0.90, 0.10]])`

In [ ]:
from GalaxySpectrumClassifier import to_xy

X_test, y_test = to_xy(test_dataset)

class_probabilities = classifier.predict_proba(X_test)
class_probabilities

## 3. Drive inference from a config file

To run inference from a script without touching code, put the constructor arguments in a small YAML file and use `ClassifierInference.from_config`. A relative `model_path` is resolved against the config file's own directory, not the working directory.

In [ ]:
from pathlib import Path

import yaml

from GalaxySpectrumClassifier import ClassifierInference

inference_config = Path("configs/random_forest_classifier_inference.yaml")
inference_config.parent.mkdir(exist_ok=True)
inference_config.write_text(
    yaml.dump(
        {
            "model_path": "../../training/binaryclassifier_simple_example/trained_random_forest/model.skops",
            "model_format": "skops",
            "task": "binary-classification",
        }
    )
)

classifier_from_config = ClassifierInference.from_config(str(inference_config))

# dataset
with open("../configs/binary_classsifier_simple_example.yaml") as f:
    config = yaml.safe_load(f)

dataset = TabularDataset.from_config(config["dataset"])
train_dataset, test_dataset = random_split(
    dataset, [0.8, 0.2], generator=torch.Generator().manual_seed(42)
)

# dataset is small and simple, we can load it into memory
X_test, y_test = to_xy(test_dataset)

predictions = classifier.predict(X_test)
accuracy = (predictions == y_test).mean()
accuracy

## 4. Torch models exported by `EpochTrainer`

`EpochTrainer.export_model` writes a directory, not a single file: `model.yaml` (the architecture manifest) plus the weights. Point `model_path` at that directory and set `model_format` to the export format — `default` for skorch's parameter format, `pt` for a plain state dict. `device` selects where the reconstructed model runs and overrides the device saved at training time.

## Train a model first for 5 epochs

In [ ]:
import yaml

from GalaxySpectrumClassifier import EpochTrainer


def to_float32(batch: dict) -> dict:
    batch = dict(batch)
    batch["source"] = [float(value) for value in batch["source"]]
    return batch


with open("../configs/binary_classifier_epoch_example.yaml") as f:
    epoch_config = yaml.safe_load(f)
epoch_config["trainer"]["progressbar"] = False

epoch_trainer = EpochTrainer.from_config(epoch_config["trainer"])
epoch_trainer.train()
epoch_trainer.export_model("exported_mlp")

We can then use it the same way as before

In [ ]:
import torch

from GalaxySpectrumClassifier import ClassifierInference, TabularDataset, to_xy

mlp_classifier = ClassifierInference(
    "../training/binaryclassifier_epoch_example/exported_mlp",
    model_format="default",
    task="binary-classification",
)

dataset = TabularDataset(
    path="../data/gold/epoch_trainer_example/test",
    data_format="csv",
    label_columns="source",
)

X_epoch, y_epoch = to_xy(dataset)

# A torch tensor is passed straight through to the skorch model.
mlp_classifier.predict(torch.as_tensor(X_epoch))[:10]

We can also iteratively apply the model to the dataset: 

In [ ]:
import numpy as np

from GalaxySpectrumClassifier import ClassifierInference, TabularDataset

mlp_classifier = ClassifierInference(
    "../training/binaryclassifier_epoch_example/exported_mlp",
    model_format="default",
    task="binary-classification",
)

dataset = TabularDataset(
    path="../data/gold/epoch_trainer_example/test",
    data_format="csv",
    label_columns="source",
)

predictions = []
accuracies = []
for x, y in dataset:
    pred = mlp_classifier.predict(x)
    acc = y == pred
    accuracies.append(acc)

acc = np.array(accuracies, dtype=np.float64)

# five number summary
percentile = np.percentile(acc, q=[0, 25, 50, 75, 100])
mean = np.mean(acc)

mean, percentile

We can also use predict_proba to get class probabilities 

In [ ]:
from GalaxySpectrumClassifier import ClassifierInference, TabularDataset, to_xy

mlp_classifier = ClassifierInference(
    "../training/binaryclassifier_epoch_example/exported_mlp",
    model_format="default",
    task="binary-classification",
)

dataset = TabularDataset(
    path="../data/gold/epoch_trainer_example/test",
    data_format="csv",
    label_columns="source",
)
train_dataset, test_dataset = random_split(
    dataset, [0.8, 0.2], generator=torch.Generator().manual_seed(42)
)
X_test, y_test = to_xy(test_dataset)

class_probabilities = classifier.predict_proba(X_test)

class_probabilities[0:10, :]

## 5. Regression and multiclass models

Regression uses `RegressionInference(model_path, model_format, device=...)`. It takes no `task` and no `classes`; `predict` returns continuous values.

Multiclass inference needs a few more details than the binary example, especially when the exported estimator already has its own non-zero-based labels. The following cells train and export a real three-class model, load it directly and from YAML, inspect probabilities, and finally exercise custom label mapping.


### 5.1 Multiclass model exported by `SimpleTrainer`

This example deliberately trains on the native labels `10`, `20`, and `30` rather than `0`, `1`, and `2`. That lets the example verify that inference preserves an exported sklearn model's real class labels instead of assuming that every prediction is a zero-based array index.


In [ ]:
import numpy as np
import torch
from sklearn.datasets import make_classification
from torch.utils.data import TensorDataset

from GalaxySpectrumClassifier import SimpleTrainer

X_multiclass, y_index = make_classification(
    n_samples=300,
    n_features=8,
    n_informative=6,
    n_redundant=0,
    n_classes=3,
    n_clusters_per_class=1,
    random_state=42,
)
native_classes = np.array([10, 20, 30])
y_multiclass = native_classes[y_index]

multiclass_dataset = TensorDataset(
    torch.as_tensor(X_multiclass, dtype=torch.float32),
    torch.as_tensor(y_multiclass),
)
multiclass_trainer = SimpleTrainer(
    output_path="../training/multiclass_inference_example",
    model_type="sklearn.linear_model.LogisticRegression",
    model_kwargs={"max_iter": 500},
    task="multiclass-classification",
)
multiclass_trainer.fit(multiclass_dataset)
multiclass_model_file = (
    "../training/multiclass_inference_example/multiclass_logistic_regression.skops"
)
multiclass_trainer.export_model(multiclass_model_file)

### 5.2 Load, predict, and inspect class probabilities

Without a `classes` override, the wrapper should preserve the estimator's native labels. For probabilities, column `i` corresponds to `multiclass_classifier.model_.classes_[i]`.


In [ ]:
from GalaxySpectrumClassifier import ClassifierInference

multiclass_classifier = ClassifierInference(
    multiclass_model_file,
    model_format="skops",
    task="multiclass-classification",
)

multiclass_predictions = multiclass_classifier.predict(X_multiclass)
multiclass_probabilities = multiclass_classifier.predict_proba(X_multiclass)
multiclass_accuracy = (multiclass_predictions == y_multiclass).mean()

(
    multiclass_accuracy,
    np.unique(multiclass_predictions),
    multiclass_classifier.model_.classes_,
    multiclass_probabilities[:3],
    multiclass_probabilities[:3].sum(axis=1),
)

### 5.3 Load the multiclass model from YAML

As in the binary example, a relative `model_path` is resolved from the inference YAML's directory.


In [ ]:
from pathlib import Path

import yaml

multiclass_inference_config = Path("configs/multiclass_inference.yaml")
multiclass_inference_config.parent.mkdir(exist_ok=True)
multiclass_inference_config.write_text(
    yaml.dump(
        {
            "model_path": "../../training/multiclass_inference_example/multiclass_logistic_regression.skops",
            "model_format": "skops",
            "task": "multiclass-classification",
        }
    )
)

multiclass_from_config = ClassifierInference.from_config(
    str(multiclass_inference_config)
)
np.testing.assert_array_equal(
    multiclass_from_config.predict(X_multiclass), multiclass_predictions
)

### 5.4 Map native classes onto application labels

The intended contract is to map the estimator's native classes `10`, `20`, and `30` onto application labels. The next cell is intentionally a regression reproducer for review finding **H2**. On the current branch it fails with `IndexError` because `ClassifierInference.predict` incorrectly treats native predictions as zero-based indices. Once H2 is fixed, the assertions below should pass without changing this example.


In [ ]:
application_classes = np.array(["star", "galaxy", "quasar"])
multiclass_with_labels = ClassifierInference(
    multiclass_model_file,
    model_format="skops",
    task="multiclass-classification",
    classes=application_classes.tolist(),
)

# Expected mapping: 10 -> star, 20 -> galaxy, 30 -> quasar.
expected_application_labels = application_classes[
    np.searchsorted(native_classes, multiclass_predictions)
]
application_predictions = multiclass_with_labels.predict(X_multiclass)

np.testing.assert_array_equal(application_predictions, expected_application_labels)
np.testing.assert_allclose(
    multiclass_with_labels.predict_proba(X_multiclass),
    multiclass_probabilities,
)